## Step 0: Install Dependencies

In [ ]:
!pip install opencv-python-headless torch torchvision Pillow matplotlib numpy scikit-image -q

## Step 1: Imports

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import os
from skimage import measure, morphology

# Device config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Step 2: Load ResNet Feature Extractor
We use a pretrained ResNet-50 (up to the last pooling layer) to extract 2048-dim embeddings per character segment. This can be swapped for a fine-tuned model later.

In [ ]:
def build_resnet_extractor():
    """
    Returns ResNet-50 with the classification head removed.
    Output: (batch, 2048) feature vectors.
    """
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    # Remove final FC layer → use as feature extractor
    extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
    extractor.eval().to(DEVICE)
    return extractor

resnet_extractor = build_resnet_extractor()
print('ResNet-50 feature extractor ready.')

# Transform for ResNet input
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),   # binary → 3-channel
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## Step 3: Image Preprocessing
Cleans up the binary image: removes noise, fills small holes, and ensures characters are dark-on-light (standard for segmentation).

In [ ]:
def preprocess_binary_image(image_path: str, 
                             invert_if_needed: bool = True,
                             noise_removal_kernel: int = 3,
                             morph_close_iter: int = 2) -> np.ndarray:
    """
    Load and clean a binary handwriting image.
    
    Parameters
    ----------
    image_path         : path to the binary image file
    invert_if_needed   : auto-detect and flip polarity so text = white on black
    noise_removal_kernel: size of kernel for morphological noise removal
    morph_close_iter   : iterations for morphological closing (fill small gaps)

    Returns
    -------
    binary : uint8 ndarray with text=255, background=0
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f'Could not load image: {image_path}')

    # Threshold (handle already-binary or near-binary input)
    _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    # Ensure text is WHITE (255), background is BLACK (0)
    if invert_if_needed:
        white_pixels = np.sum(binary == 255)
        black_pixels = np.sum(binary == 0)
        if white_pixels > black_pixels:
            # More white than black → background is white → invert
            binary = cv2.bitwise_not(binary)

    # Remove small noise blobs
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT,
                                       (noise_removal_kernel, noise_removal_kernel))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Close small gaps within characters
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel,
                               iterations=morph_close_iter)

    return binary


def visualise(img: np.ndarray, title: str = '', cmap: str = 'gray'):
    plt.figure(figsize=(12, 4))
    plt.imshow(img, cmap=cmap)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print('Preprocessing functions defined.')

## Step 4: Character Segmentation
Two complementary strategies:
- **Connected Components** — fast, works well when characters are clearly separated
- **Contour-based** — more robust for slightly touching characters

In [ ]:
def segment_connected_components(binary: np.ndarray,
                                  min_area: int = 50,
                                  max_area: int = None,
                                  padding: int = 5) -> list:
    """
    Segment characters via connected components analysis.

    Returns list of dicts:
        {'bbox': (x, y, w, h), 'crop': uint8 ndarray, 'area': int}
    sorted left-to-right, top-to-bottom.
    """
    h_img, w_img = binary.shape
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8)

    segments = []
    for label in range(1, num_labels):          # skip label 0 (background)
        x, y, w, h, area = stats[label]
        if area < min_area:
            continue
        if max_area and area > max_area:
            continue

        # Add padding, clamp to image bounds
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(w_img, x + w + padding)
        y2 = min(h_img, y + h + padding)

        crop = binary[y1:y2, x1:x2].copy()
        segments.append({
            'bbox': (x1, y1, x2 - x1, y2 - y1),
            'crop': crop,
            'area': int(area),
            'centroid': centroids[label]
        })

    # Sort: top-to-bottom, left-to-right (reading order)
    segments.sort(key=lambda s: (s['bbox'][1] // 30, s['bbox'][0]))
    return segments


def segment_contours(binary: np.ndarray,
                      min_area: int = 50,
                      padding: int = 5) -> list:
    """
    Segment characters via external contour detection.
    More tolerant of slight character touches.
    """
    h_img, w_img = binary.shape
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    segments = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)

        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(w_img, x + w + padding)
        y2 = min(h_img, y + h + padding)

        crop = binary[y1:y2, x1:x2].copy()
        segments.append({
            'bbox': (x1, y1, x2 - x1, y2 - y1),
            'crop': crop,
            'area': int(area)
        })

    segments.sort(key=lambda s: (s['bbox'][1] // 30, s['bbox'][0]))
    return segments


print('Segmentation functions defined.')

## Step 5: ResNet Feature Extraction per Segment

In [ ]:
@torch.no_grad()
def extract_features_batch(segments: list,
                             model,
                             transform,
                             batch_size: int = 32) -> np.ndarray:
    """
    Run ResNet on all character crops in batches.

    Returns
    -------
    features : ndarray of shape (N, 2048)
    """
    all_features = []
    crops = [seg['crop'] for seg in segments]

    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i + batch_size]
        tensors = []
        for crop in batch_crops:
            pil_img = Image.fromarray(crop)
            tensors.append(transform(pil_img))

        batch_tensor = torch.stack(tensors).to(DEVICE)
        feats = model(batch_tensor)                    # (B, 2048, 1, 1)
        feats = feats.squeeze(-1).squeeze(-1).cpu().numpy()  # (B, 2048)
        all_features.append(feats)

    return np.concatenate(all_features, axis=0) if all_features else np.array([])


print('Feature extraction function defined.')

## Step 6: Visualisation Helpers

In [ ]:
def draw_bounding_boxes(original_binary: np.ndarray, segments: list) -> np.ndarray:
    """Draw green bounding boxes on the image for all detected segments."""
    vis = cv2.cvtColor(original_binary, cv2.COLOR_GRAY2BGR)
    for seg in segments:
        x, y, w, h = seg['bbox']
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 220, 100), 2)
    return vis


def show_character_grid(segments: list, max_chars: int = 60,
                         cols: int = 10, char_size: int = 64):
    """
    Display a grid of all segmented character crops.
    """
    n = min(len(segments), max_chars)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))

    if rows == 1:
        axes = [axes] if cols == 1 else list(axes)
        axes = [axes]

    idx = 0
    for r in range(rows):
        for c in range(cols):
            ax = axes[r][c] if rows > 1 else axes[0][c]
            if idx < n:
                crop_resized = cv2.resize(segments[idx]['crop'],
                                          (char_size, char_size),
                                          interpolation=cv2.INTER_NEAREST)
                ax.imshow(crop_resized, cmap='gray')
                ax.set_title(f'#{idx}', fontsize=7)
            ax.axis('off')
            idx += 1

    plt.suptitle(f'Segmented Characters ({n} shown)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


print('Visualisation helpers defined.')

## Step 7: Save Segmented Characters

In [ ]:
def save_segments(segments: list, output_dir: str = 'segmented_chars',
                   prefix: str = 'char'):
    """
    Save each character crop as a PNG file.
    Files are named: <prefix>_0000.png, <prefix>_0001.png …
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    for i, seg in enumerate(segments):
        filename = os.path.join(output_dir, f'{prefix}_{i:04d}.png')
        cv2.imwrite(filename, seg['crop'])
    print(f'Saved {len(segments)} character images → "{output_dir}"')


print('Save function defined.')

## Step 8: Full Pipeline — Run Everything
**👉 Change `IMAGE_PATH` to your binary image file.**

In [ ]:
# ─────────────────────────────────────────────
#  USER CONFIGURATION — edit these values
# ─────────────────────────────────────────────
IMAGE_PATH       = 'your_handwriting.png'   # ← path to your binary image
OUTPUT_DIR       = 'segmented_chars'         # where to save individual char images
SEGMENTATION_MODE= 'connected_components'    # 'connected_components' or 'contours'
MIN_AREA         = 80                        # ignore blobs smaller than this (pixels²)
PADDING          = 6                         # extra pixels around each character bbox
EXTRACT_FEATURES = True                      # set False to skip ResNet (faster)
# ─────────────────────────────────────────────


# --- Stage 1: Preprocess ---
print('=== Stage 1: Preprocessing ===')
binary = preprocess_binary_image(IMAGE_PATH)
visualise(binary, 'Preprocessed Binary Image')


# --- Stage 2: Segment ---
print(f'\n=== Stage 2: Segmentation ({SEGMENTATION_MODE}) ===')
if SEGMENTATION_MODE == 'connected_components':
    segments = segment_connected_components(binary,
                                             min_area=MIN_AREA,
                                             padding=PADDING)
else:
    segments = segment_contours(binary,
                                  min_area=MIN_AREA,
                                  padding=PADDING)

print(f'Found {len(segments)} character segments.')


# --- Stage 3: Visualise bounding boxes ---
print('\n=== Stage 3: Bounding Boxes ===')
vis_img = draw_bounding_boxes(binary, segments)
plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title(f'Detected Segments: {len(segments)}', fontsize=14)
plt.axis('off')
plt.show()


# --- Stage 4: Show character grid ---
print('\n=== Stage 4: Character Grid ===')
show_character_grid(segments)


# --- Stage 5: ResNet feature extraction ---
if EXTRACT_FEATURES and len(segments) > 0:
    print('\n=== Stage 5: ResNet-50 Feature Extraction ===')
    features = extract_features_batch(segments, resnet_extractor, resnet_transform)
    print(f'Feature matrix shape: {features.shape}  '
          f'(N chars × 2048 ResNet dims)')

    # Attach features back to segment dicts for downstream use
    for i, seg in enumerate(segments):
        seg['feature'] = features[i]
else:
    features = None
    print('\nSkipping ResNet feature extraction.')


# --- Stage 6: Save individual character images ---
print('\n=== Stage 6: Saving Character Images ===')
save_segments(segments, output_dir=OUTPUT_DIR)

print('\n✅ Pipeline complete!')

## Step 9 (Optional): UMAP Visualisation of ResNet Embeddings
If you have ground-truth labels, colour the points to see how well ResNet separates characters.

In [ ]:
# Requires: pip install umap-learn
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('umap-learn not installed — run: pip install umap-learn')


if UMAP_AVAILABLE and features is not None and len(features) >= 10:
    reducer = umap.UMAP(n_neighbors=min(15, len(features) - 1),
                         random_state=42)
    embedding = reducer.fit_transform(features)

    plt.figure(figsize=(8, 6))
    plt.scatter(embedding[:, 0], embedding[:, 1],
                c=range(len(embedding)), cmap='tab20',
                s=40, alpha=0.8)
    plt.title('UMAP of ResNet-50 Character Embeddings', fontsize=13)
    plt.colorbar(label='Character index')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping UMAP — either unavailable, no features, or too few segments.')

## Step 10 (Optional): Fine-Tune ResNet on Your Own Character Labels
Attach a classification head and fine-tune on your labelled crops.

In [ ]:
def build_resnet_classifier(num_classes: int):
    """
    Full ResNet-50 with a custom head for `num_classes` character classes.
    Freeze all layers except the final FC for quick fine-tuning.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Freeze backbone
    for param in model.parameters():
        param.requires_grad = False

    # Replace head
    model.fc = torch.nn.Sequential(
        torch.nn.Linear(2048, 512),
        torch.nn.ReLU(),
        torch.nn.Dropout(0.4),
        torch.nn.Linear(512, num_classes)
    )
    return model.to(DEVICE)


# Example — uncomment and adapt once you have labelled data:
# NUM_CLASSES = 62   # e.g. 26 upper + 26 lower + 10 digits
# classifier  = build_resnet_classifier(NUM_CLASSES)
# optimizer   = torch.optim.Adam(classifier.fc.parameters(), lr=1e-3)
# criterion   = torch.nn.CrossEntropyLoss()
print('Fine-tuning scaffold ready (uncomment to use with labelled data).')